## Make csv inbox volume

In [0]:
%sql
USE CATALOG predictivemaintenancecatalog;
USE SCHEMA raw_data;

CREATE VOLUME IF NOT EXISTS landing

In [0]:
csv_inbox = "/Volumes/predictivemaintenancecatalog/raw_data/landing/csv_inbox"

dbutils.fs.mkdirs(csv_inbox)

display(dbutils.fs.ls(csv_inbox))

## Inspect the Landing Delta Table
This we we are going to finally drop the the csv too. But first we are going to put it in the csv_inbox

In [0]:
landing_table = "predictivemaintenancecatalog.raw_data.landing_testdata"

df_landing = spark.table(landing_table)

df_landing.printSchema()

display(df_landing.limit(5))

In [0]:
display(dbutils.fs.ls(csv_inbox))

## Step 3 — Configure Auto Loader

### Read incoming CSV files

In [0]:
landing_table = "predictivemaintenancecatalog.raw_data.landing_testdata"

landing_path = "/Volumes/predictivemaintenancecatalog/raw_data/landing"
csv_inbox = f"{landing_path}/csv_inbox"

schema_location = f"{landing_path}/_schemas/csv_autoloader"
checkpoint_location = f"{landing_path}/_checkpoints/csv_autoloader"

# Use the schema of our existing landing table
landing_schema = spark.table(landing_table).schema

# Configure Auto Loader
csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
    .schema(landing_schema)
    .load(csv_inbox)
    .select(*landing_schema.fieldNames())
)

### Write to the existing landing table

In [0]:
query = (
    csv_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(landing_table)
)

query.awaitTermination()

## Step 4 — Verify ingestion

In [0]:
spark.sql(f"""
    SELECT
        MIN(UDI) AS min_udi,
        MAX(UDI) AS max_udi,
        COUNT(*) AS total_records
    FROM {landing_table}
    WHERE UDI BETWEEN 1000001 AND 1000500
""").show()

In [0]:
display(spark.table("predictivemaintenancecatalog.raw_data.landing_testdata"))